# Basic Usage Examples

This notebook provides simple, working examples to get you started with POMDPPlanners quickly.

## Overview

**Planning Algorithms:**
- **POMCP**: Monte Carlo Tree Search with particle belief updates — for discrete action and observation spaces
- **PFT-DPW**: Particle Filter Tree with Double Progressive Widening — for continuous action and observation spaces

**Environments:**
- **Tiger POMDP**: Classic discrete POMDP benchmark with two actions (open door / listen)
- **Light-Dark POMDP**: Continuous navigation where observation noise depends on position

**Key Concepts Demonstrated:**
- The fundamental POMDP loop: plan → act → observe → update belief
- Using `run_episode()` for streamlined episode execution with history recording
- Working with continuous action spaces and custom action samplers
- Using vectorized particle beliefs for efficient batched belief updates
- Visualizing episode trajectories as animated GIFs
- Implementing custom planners by extending the `Policy` base class

## Your First POMDP Solution

The core of any POMDP problem is a loop: **plan → act → observe → update belief → repeat**.
Let's implement this manually with the Tiger POMDP to see each step clearly.

The Tiger POMDP is a classic discrete benchmark: a tiger hides behind one of two doors. The agent can **listen** (noisy observation, small cost) or **open a door** (large reward if correct, large penalty if wrong). The challenge is gathering enough information before committing to an irreversible action.

In [12]:
from POMDPPlanners.environments.tiger_pomdp import TigerPOMDP
from POMDPPlanners.planners.mcts_planners.pomcp import POMCP
from POMDPPlanners.core.belief import get_initial_belief
import numpy as np

# Create the environment and planner
env = TigerPOMDP(discount_factor=0.95)
planner = POMCP(
    environment=env,
    discount_factor=0.95,
    depth=10,
    exploration_constant=50.0,
    name="tiger_planner",
    n_simulations=100,
)

# Initialize state and belief
state = env.initial_state_dist().sample()[0]
belief = get_initial_belief(env, n_particles=200)

print(f"Initial state: {state}\n")

# Manual POMDP loop: plan → act → observe → update belief
num_steps = 10
total_reward = 0.0

for step in range(num_steps):
    # Plan: select action from current belief
    actions, _ = planner.action(belief)
    action = actions[0]

    # Act: step the environment
    next_state, observation, reward = env.sample_next_step(state, action)

    # Update belief with the action and received observation
    belief = belief.update(action=action, observation=observation, pomdp=env)

    state = next_state
    total_reward += reward
    print(f"Step {step + 1:2d}: action={str(action):<20} obs={str(observation):<20} reward={reward:.1f}")

print(f"\nTotal reward: {total_reward:.1f}")

# Inspect the final belief: it's a weighted particle filter over possible states
weights = np.exp(belief.log_weights)
print(f"\nMost likely state: {belief.particles[np.argmax(weights)]}")
print(f"Particle sample: {belief.particles[:5]}")

2026-02-24 22:10:14,756 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing TigerPOMDP environment with discount factor 0.95
2026-02-24 22:10:14,757 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: tiger_planner (debug=False)


Initial state: tiger_right

Step  1: action=listen               obs=hear_right           reward=-1.0
Step  2: action=open_left            obs=hear_nothing         reward=10.0
Step  3: action=open_left            obs=hear_nothing         reward=-100.0
Step  4: action=listen               obs=hear_right           reward=-1.0
Step  5: action=listen               obs=hear_left            reward=-1.0
Step  6: action=listen               obs=hear_right           reward=-1.0
Step  7: action=open_left            obs=hear_nothing         reward=10.0
Step  8: action=open_left            obs=hear_nothing         reward=-100.0
Step  9: action=open_left            obs=hear_nothing         reward=-100.0
Step 10: action=open_right           obs=hear_nothing         reward=10.0

Total reward: -274.0

Most likely state: tiger_left
Particle sample: ['tiger_left', 'tiger_left', 'tiger_left', 'tiger_right', 'tiger_left']


## Running a Complete Episode

The loop above is exactly what `run_episode()` does internally. Here's how to run the same experiment using the package's built-in episode runner, which also handles logging, history recording, and timing:

In [13]:
from POMDPPlanners.simulations.episodes import run_episode
from POMDPPlanners.core.simulation import history_to_discounted_return_value

# run_episode handles the plan-act-observe-update loop and records timing
history = run_episode(
    environment=env,
    policy=planner,
    initial_belief=get_initial_belief(env, n_particles=200),
    num_steps=10,
    logger=None,
)

# Inspect the history
print(f"Steps executed: {history.actual_num_steps}")
print(f"Reached terminal state: {history.reach_terminal_state}")
print(f"Discounted return: {history_to_discounted_return_value(history):.2f}")
print(f"Avg action selection time: {history.average_action_time:.4f}s")
print(f"Avg belief update time:    {history.average_belief_update_time:.4f}s")

# Step-by-step access
print("\nStep-by-step replay:")
for i, step in enumerate(history.history):
    if step.reward is not None:
        print(f"  Step {i + 1}: action={str(step.action):<20} obs={str(step.observation):<20} reward={step.reward:.1f}")

Steps executed: 10
Reached terminal state: False
Discounted return: -73.80
Avg action selection time: 0.0210s
Avg belief update time:    0.0007s

Step-by-step replay:
  Step 1: action=listen               obs=hear_right           reward=-1.0
  Step 2: action=listen               obs=hear_left            reward=-1.0
  Step 3: action=open_right           obs=hear_nothing         reward=-100.0
  Step 4: action=listen               obs=hear_left            reward=-1.0
  Step 5: action=open_right           obs=hear_nothing         reward=10.0
  Step 6: action=listen               obs=hear_right           reward=-1.0
  Step 7: action=listen               obs=hear_right           reward=-1.0
  Step 8: action=open_left            obs=hear_nothing         reward=10.0
  Step 9: action=listen               obs=hear_right           reward=-1.0
  Step 10: action=open_left            obs=hear_nothing         reward=10.0


## Continuous Action Spaces: Light-Dark POMDP

When actions are continuous vectors rather than discrete choices, two things change:

1. **Planner**: POMCP enumerates actions, so it doesn't work. PFT-DPW uses an `ActionSampler` to generate candidate actions during planning instead.
2. **Action sampler**: You provide a custom `ActionSampler` that defines how actions are drawn from the continuous space.

The Light-Dark POMDP is a navigation task where the agent must move to a goal position. Observations are noisy position estimates — the closer to the "light" region, the lower the noise. The agent must first move into the light to localize itself, then navigate to the goal.

This example also uses `create_environment_belief` from the belief factory, which automatically selects a **vectorized particle belief** for environments that support it. Vectorized beliefs perform batched NumPy transitions and observation-likelihood computations, avoiding Python-level loops over particles.

In [26]:
from POMDPPlanners.environments.light_dark_pomdp.continuous_light_dark_pomdp import (
    ContinuousLightDarkPOMDP, RewardModelType
)
from POMDPPlanners.planners.mcts_planners.pft_dpw import PFT_DPW
from POMDPPlanners.planners.planners_utils.dpw import ActionSampler
from POMDPPlanners.utils.belief_factory import create_environment_belief
import numpy as np

# Create navigation environment
env = ContinuousLightDarkPOMDP(
    discount_factor=0.95,
    goal_state=np.array([10, 5]),
    start_state=np.array([0, 5]),
    reward_model_type=RewardModelType.STANDARD,
    obstacles=[]
)

# Action sampler for continuous actions
class NavigationActionSampler(ActionSampler):
    def sample(self, belief_node=None):
        # Sample 2D unit vector (direction only, fixed speed)
        angle = np.random.uniform(0, 2 * np.pi)
        return np.array([np.cos(angle), np.sin(angle)])

# Use PFT_DPW for continuous navigation
planner = PFT_DPW(
    environment=env,
    discount_factor=0.95,
    depth=5,
    name="navigation_planner",
    action_sampler=NavigationActionSampler(),
    n_simulations=3000,
    k_a=8,
    alpha_a=0.01,
    k_o=10,
    alpha_o=0.01,
    exploration_constant=1000,
)

# Run a short navigation episode
state = env.initial_state_dist().sample()[0]

# Use vectorized particle belief for faster belief updates
belief = create_environment_belief(env, n_particles=200)

print(f"Start position: {state[:2]}")
print(f"Goal position:  {env.goal_state}")
print(f"Belief type:    {type(belief).__name__}\n")

for step in range(5):
    actions, _ = planner.action(belief)
    action = actions[0]

    next_state, observation, reward = env.sample_next_step(state, action)
    belief = belief.update(action=action, observation=observation, pomdp=env)

    print(f"Step {step + 1}: action=[{action[0]:+.2f}, {action[1]:+.2f}]  "
          f"pos=[{next_state[0]:.2f}, {next_state[1]:.2f}]  reward={reward:.2f}")
    state = next_state

2026-02-24 22:25:33,176 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing ContinuousLightDarkPOMDP environment with discount factor 0.95
2026-02-24 22:25:33,177 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: navigation_planner (debug=False)


Start position: [0 5]
Goal position:  [10  5]
Belief type:    VectorizedWeightedParticleBelief

Step 1: action=[+0.78, -0.63]  pos=[0.29, 2.49]  reward=-11.24
Step 2: action=[+0.15, +0.99]  pos=[-0.10, 3.25]  reward=-11.68
Step 3: action=[+0.83, -0.56]  pos=[0.36, 2.46]  reward=-11.55
Step 4: action=[+0.94, -0.33]  pos=[1.41, 3.86]  reward=-11.15
Step 5: action=[+0.52, +0.85]  pos=[2.69, 5.21]  reward=-10.07


## Visualization Example

`visualize_planner_episode` runs multiple full episodes and saves each one as an animated GIF. Each frame shows the current state, the agent's action, and how the belief evolves — useful for quickly checking whether a planner is behaving sensibly.

In [27]:
from pathlib import Path
from POMDPPlanners.utils.planner_episode_visualization import visualize_planner_episode

# Reuse the Light-Dark environment and PFT-DPW planner from above
visualize_planner_episode(
    planner=planner,
    environment=env,
    belief=create_environment_belief(env, n_particles=200),
    n_episodes=1,
    cache_dir=Path("cache/visualizations"),
    num_steps=20,
)
print("GIF saved to cache/visualizations/")

GIF saved to cache/visualizations/


## Custom Planners

All planners in POMDPPlanners must inherit from the `Policy` class in `POMDPPlanners.core.policy`. The key method to implement is `action()`, which returns a list of actions to be executed sequentially:

- **Closed-loop planning**: When the list contains one action (length = 1)
- **Open-loop planning**: When the list contains multiple actions (length > 1)

### Key Requirements

1. **Inherit from Policy**: All planners must extend the abstract `Policy` class
2. **Implement action()**: Returns `(List[Action], PolicyRunData)` tuple
3. **Implement get_space_info()**: Declares what environments the planner can solve

### Simple RandomPlanner Example

Here's a minimal implementation of a random action planner:

In [28]:
import random
from typing import Any, List, Tuple

from POMDPPlanners.core.belief import Belief
from POMDPPlanners.core.environment import SpaceType
from POMDPPlanners.core.policy import Policy, PolicyRunData, PolicySpaceInfo


class RandomPlanner(Policy):
    """Planner that selects actions uniformly at random from discrete action spaces."""

    def action(self, belief: Belief) -> Tuple[List[Any], PolicyRunData]:
        actions = self.environment.get_actions()  # type: ignore[attr-defined]
        chosen = random.choice(actions)
        return [chosen], PolicyRunData(info_variables=[])

    @classmethod
    def get_space_info(cls) -> PolicySpaceInfo:
        return PolicySpaceInfo(
            action_space=SpaceType.DISCRETE,
            observation_space=SpaceType.MIXED,  # works with any observation type
        )

    @classmethod
    def get_info_variable_names(cls) -> List[str]:
        return []


# Try it on Tiger POMDP
env = TigerPOMDP(discount_factor=0.95)
random_planner = RandomPlanner(
    environment=env,
    discount_factor=0.95,
    name="random_planner",
)

belief = get_initial_belief(env, n_particles=200)
history = run_episode(
    environment=env,
    policy=random_planner,
    initial_belief=belief,
    num_steps=10,
    logger=None,
)

print(f"Random planner discounted return: {history_to_discounted_return_value(history):.2f}")
print(f"Steps: {history.actual_num_steps}")
for i, step in enumerate(history.history):
    if step.reward is not None:
        print(f"  Step {i + 1}: action={str(step.action):<20} reward={step.reward:.1f}")

2026-02-24 22:26:35,292 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/environment.py:296 - Initializing TigerPOMDP environment with discount factor 0.95
2026-02-24 22:26:35,293 - INFO: /home/kobi/Documents/github/POMDPPlanners/POMDPPlanners/core/policy.py:306 - Initialized policy: random_planner (debug=False)


Random planner discounted return: -263.57
Steps: 10
  Step 1: action=open_left            reward=10.0
  Step 2: action=open_left            reward=10.0
  Step 3: action=listen               reward=-1.0
  Step 4: action=open_right           reward=-100.0
  Step 5: action=open_left            reward=-100.0
  Step 6: action=open_left            reward=10.0
  Step 7: action=open_right           reward=10.0
  Step 8: action=listen               reward=-1.0
  Step 9: action=open_left            reward=-100.0
  Step 10: action=open_left            reward=-100.0


## What's Next?

- **Compare planners at scale**: See [planners_comparison.ipynb](planners_comparison.ipynb) for running batch evaluations across multiple algorithms and environments with statistical analysis
- **Belief representations**: See [belief_representations.ipynb](belief_representations.ipynb) for particle filters, Gaussian beliefs, Kalman filter variants, and Gaussian mixtures
- **Build your own environment**: See [custom_environment.ipynb](custom_environment.ipynb) to create a POMDP from scratch and run planners on it
- **Inspect planner behavior**: See [tree_analysis_debugging.ipynb](tree_analysis_debugging.ipynb) for MCTS tree metrics, visualization, and debugging workflows
- **Explore more environments**: The package includes CartPole, Mountain Car, Push, and other POMDP environments — each follows the same `Environment` interface shown above
- **Tune hyperparameters**: Use `LocalSimulationsAPI` to sweep over planner configurations and find the best settings for your problem